# Solutions: Text Data Preparation for LLMs

This notebook contains possible solutions for the practice exercises in `Data Preparation for LLMs.ipynb`. The goal is not to memorize one exact answer, but to see clear Python patterns for cleaning, filtering, chunking, and preparing text for LLM workflows.

## Exercise 1 solution: Clean a list of text snippets

This solution strips whitespace, collapses repeated spaces, redacts a sample email address, and skips empty strings.

In [1]:
exercise_texts = [
    "  Prompt engineering needs clean inputs.  ",
    "",
    "LLM data should not include private emails like teacher@example.com.",
]

cleaned_exercise_texts = []

for text in exercise_texts:
    cleaned_text = text.strip()
    cleaned_text = " ".join(cleaned_text.split())
    cleaned_text = cleaned_text.replace("teacher@example.com", "[EMAIL]")

    if not cleaned_text:
        continue

    cleaned_exercise_texts.append(cleaned_text)

print(cleaned_exercise_texts)

['Prompt engineering needs clean inputs.', 'LLM data should not include private emails like [EMAIL].']


## Exercise 2 solution: Chunk a document

This solution splits the text into words, slices the words six at a time, and joins each slice back into a string.

In [2]:
exercise_document = "LLMs work better when the input text is clean organized and connected to useful metadata."
exercise_chunk_size = 6
exercise_chunks = []

words = exercise_document.split()

for start in range(0, len(words), exercise_chunk_size):
    word_chunk = words[start:start + exercise_chunk_size]
    exercise_chunks.append(" ".join(word_chunk))

print(exercise_chunks)

['LLMs work better when the input', 'text is clean organized and connected', 'to useful metadata.']


## Exercise 3 solution: Combine text with metadata using `zip()`

This solution reads matching values from two lists and stores each pair as a dictionary.

In [3]:
topics = ["cleaning", "filtering", "chunking"]
texts = [
    "Remove unnecessary whitespace.",
    "Skip records that are empty or too short.",
    "Split long text into smaller pieces.",
]

training_records = []

for topic, text in zip(topics, texts):
    training_records.append({
        "topic": topic,
        "text": text,
    })

print(training_records)

[{'topic': 'cleaning', 'text': 'Remove unnecessary whitespace.'}, {'topic': 'filtering', 'text': 'Skip records that are empty or too short.'}, {'topic': 'chunking', 'text': 'Split long text into smaller pieces.'}]


## Exercise 4 solution: Mini project

This solution cleans each note, skips weak records, chunks valid text, and keeps the original note id as metadata.

In [4]:
raw_notes = [
    {"id": "note-1", "text": "  Lists and dictionaries are useful for storing structured records.  "},
    {"id": "note-2", "text": "short"},
    {"id": "note-3", "text": "Use loops and conditions to prepare text before sending it to a language model."},
]

prepared_records = []
minimum_length = 20
chunk_size = 6

for note in raw_notes:
    clean_text = note["text"].strip()
    clean_text = " ".join(clean_text.split())

    if len(clean_text) < minimum_length:
        continue

    words = clean_text.split()

    for start in range(0, len(words), chunk_size):
        word_chunk = words[start:start + chunk_size]
        prepared_records.append({
            "note_id": note["id"],
            "chunk_number": len(prepared_records) + 1,
            "text": " ".join(word_chunk),
            "word_count": len(word_chunk),
        })

print(prepared_records)

[{'note_id': 'note-1', 'chunk_number': 1, 'text': 'Lists and dictionaries are useful for', 'word_count': 6}, {'note_id': 'note-1', 'chunk_number': 2, 'text': 'storing structured records.', 'word_count': 3}, {'note_id': 'note-3', 'chunk_number': 3, 'text': 'Use loops and conditions to prepare', 'word_count': 6}, {'note_id': 'note-3', 'chunk_number': 4, 'text': 'text before sending it to a', 'word_count': 6}, {'note_id': 'note-3', 'chunk_number': 5, 'text': 'language model.', 'word_count': 2}]


## Challenge solution: Build a retrieval-ready mini dataset

This solution combines cleaning, redaction, filtering, chunking, metadata, a quality report, and prompt-ready output.

In [5]:
challenge_notes = [
    {
        "id": "kb-001",
        "topic": "loops",
        "priority": "high",
        "text": "  Use for loops when you already have a collection. Use while loops when you need to repeat until a condition changes.  ",
    },
    {
        "id": "kb-002",
        "topic": "privacy",
        "priority": "high",
        "text": "Contact admin@example.com before sharing notebooks that contain student data.",
    },
    {
        "id": "kb-003",
        "topic": "draft",
        "priority": "low",
        "text": "temporary note",
    },
    {
        "id": "kb-004",
        "topic": "chunking",
        "priority": "medium",
        "text": "Chunking keeps long context manageable. Each chunk should keep enough words to be useful and enough metadata to be traceable.",
    },
    {
        "id": "kb-005",
        "topic": "empty",
        "priority": "high",
        "text": "     ",
    },
]

challenge_chunk_size = 7
minimum_words = 5
challenge_chunks = []
skipped_notes = []

for note in challenge_notes:
    clean_note_text = note["text"].strip()
    clean_note_text = " ".join(clean_note_text.split())
    clean_note_text = clean_note_text.replace("admin@example.com", "[EMAIL]")
    words = clean_note_text.split()

    if note["priority"] == "low" or len(words) < minimum_words:
        skipped_notes.append(note["id"])
        continue

    chunk_number = 1

    for start in range(0, len(words), challenge_chunk_size):
        word_chunk = words[start:start + challenge_chunk_size]
        challenge_chunks.append({
            "note_id": note["id"],
            "topic": note["topic"],
            "priority": note["priority"],
            "chunk_number": chunk_number,
            "text": " ".join(word_chunk),
            "word_count": len(word_chunk),
        })
        chunk_number += 1

topics_included = sorted({chunk["topic"] for chunk in challenge_chunks})
quality_report = {
    "skipped_notes": len(skipped_notes),
    "created_chunks": len(challenge_chunks),
    "topics_included": topics_included,
}

print("Quality report:")
print(quality_report)

print("\nPrompt-ready context:")
for number, chunk in enumerate(challenge_chunks[:3], start=1):
    print(f"{number}. [{chunk['note_id']} | {chunk['topic']}] {chunk['text']}")

Quality report:
{'skipped_notes': 2, 'created_chunks': 8, 'topics_included': ['chunking', 'loops', 'privacy']}

Prompt-ready context:
1. [kb-001 | loops] Use for loops when you already have
2. [kb-001 | loops] a collection. Use while loops when you
3. [kb-001 | loops] need to repeat until a condition changes.


## Optional discussion

There are many valid solutions. For example, you could use a helper function for cleaning, store skipped-note reasons instead of only ids, or use a list comprehension for selected parts. The important parts are readable control flow, useful metadata, and checks that make the prepared text trustworthy.